# 2014 — GRU and Sequence-to-Sequence Learning

**Papers:** Kyunghyun Cho et al., *"Learning Phrase Representations using RNN Encoder–Decoder for Statistical Machine Translation"* (2014), which introduced the **GRU**; Ilya Sutskever, Oriol Vinyals & Quoc Le, *"Sequence to Sequence Learning with Neural Networks"* (2014).

**GRU** is a simpler LSTM with two gates and no separate cell state:

$$
z_t = \sigma(W_z[x_t,h_{t-1}]),\quad r_t = \sigma(W_r[x_t,h_{t-1}]),\quad \tilde h_t = \tanh(W_h[x_t,\ r_t\odot h_{t-1}]),\quad h_t = (1-z_t)\odot h_{t-1} + z_t\odot\tilde h_t
$$

**Seq2seq:** an *encoder* RNN reads the input sequence into a fixed vector, and a *decoder* RNN generates the output sequence from it. This was the first neural approach to work well for machine translation. It is trained with **teacher forcing** (the decoder is fed the true previous token) and decoded greedily.

Task: **sort** a sequence of digits, e.g. `5 1 4 1` → `1 1 4 5`. Input and output lengths can differ in general, and seq2seq handles that naturally.

In [1]:
import os, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

def get_device():
    """CUDA > Apple MPS > CPU. Force one with the env var NN_DEVICE=cpu|cuda|mps."""
    forced = os.environ.get("NN_DEVICE")
    if forced:
        return torch.device(forced)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def seed_everything(seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

device = get_device()
torch.set_flush_denormal(True)   # tiny "denormal" floats are extremely slow on CPUs; treat them as 0
seed_everything(0)
print("PyTorch", torch.__version__, "| device:", device)

PyTorch 2.15.0 | device: mps


In [2]:
class GRUCell(nn.Module):
    def __init__(self, n_in, n_hid):
        super().__init__()
        self.x2 = nn.Linear(n_in, 3 * n_hid); self.h2 = nn.Linear(n_hid, 3 * n_hid)
    def forward(self, x, h):
        xr, xz, xn = self.x2(x).chunk(3, -1); hr, hz, hn = self.h2(h).chunk(3, -1)
        r, z = torch.sigmoid(xr + hr), torch.sigmoid(xz + hz)
        n = torch.tanh(xn + r * hn)
        return (1 - z) * n + z * h          # PyTorch's convention: z gates the *old* state

cell, ref = GRUCell(4, 6), nn.GRUCell(4, 6)
with torch.no_grad():
    cell.x2.weight.copy_(ref.weight_ih); cell.x2.bias.copy_(ref.bias_ih)
    cell.h2.weight.copy_(ref.weight_hh); cell.h2.bias.copy_(ref.bias_hh)
x, h = torch.randn(3, 4), torch.randn(3, 6)
print("max difference vs nn.GRUCell:", (cell(x, h) - ref(x, h)).abs().max().item())

max difference vs nn.GRUCell: 8.940696716308594e-08


## Data: sort digits
Tokens 0–9 are digits, 10 = `<sos>` (start of output), 11 = `<eos>`.

In [3]:
SOS, EOS, V, L = 10, 11, 12, 8
def batch(B, L=L):
    x = torch.randint(0, 10, (B, L))
    y = torch.cat([x.sort(1).values, torch.full((B, 1), EOS)], 1)
    return x.to(device), y.to(device)

class Seq2Seq(nn.Module):
    def __init__(self, V, d=128):
        super().__init__()
        self.emb = nn.Embedding(V, d)
        self.encoder = nn.GRU(d, d, batch_first=True)
        self.decoder = nn.GRU(d, d, batch_first=True)
        self.out = nn.Linear(d, V)

    def forward(self, src, tgt):                       # teacher forcing
        _, h = self.encoder(self.emb(src))             # h: the whole input squeezed into one vector
        dec_in = torch.cat([torch.full_like(tgt[:, :1], SOS), tgt[:, :-1]], 1)
        o, _ = self.decoder(self.emb(dec_in), h)
        return self.out(o)

    @torch.no_grad()
    def generate(self, src, max_len):
        _, h = self.encoder(self.emb(src))
        tok = torch.full((len(src), 1), SOS, device=src.device); out = []
        for _ in range(max_len):
            o, h = self.decoder(self.emb(tok), h)
            tok = self.out(o).argmax(-1); out.append(tok)
        return torch.cat(out, 1)

model = Seq2Seq(V).to(device)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
for step in range(3001):
    x, y = batch(128)
    loss = F.cross_entropy(model(x, y).reshape(-1, V), y.reshape(-1))
    opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
    if step % 500 == 0:
        xt, yt = batch(500)
        pred = model.generate(xt, L + 1)
        print(f"step {step:4d} loss {loss.item():.3f} | token acc {(pred == yt).float().mean():.3f} | full-sequence acc {(pred == yt).all(1).float().mean():.3f}")

step    0 loss 2.479 | token acc 0.216 | full-sequence acc 0.000


step  500 loss 0.005 | token acc 1.000 | full-sequence acc 1.000


step 1000 loss 0.003 | token acc 1.000 | full-sequence acc 0.998


step 1500 loss 0.005 | token acc 0.999 | full-sequence acc 0.992


step 2000 loss 0.000 | token acc 1.000 | full-sequence acc 1.000


step 2500 loss 0.000 | token acc 1.000 | full-sequence acc 1.000


step 3000 loss 0.000 | token acc 1.000 | full-sequence acc 1.000


In [4]:
xt, yt = batch(5)
for a, b in zip(xt.tolist(), model.generate(xt, L + 1).tolist()):
    print(a, "->", b[:-1])

[1, 7, 8, 8, 5, 5, 1, 3] -> [1, 1, 3, 5, 5, 7, 8, 8]
[3, 3, 2, 1, 0, 5, 4, 4] -> [0, 1, 2, 3, 3, 4, 4, 5]
[8, 0, 6, 9, 1, 1, 0, 5] -> [0, 0, 1, 1, 5, 6, 8, 9]
[9, 2, 3, 6, 9, 6, 3, 4] -> [2, 3, 3, 4, 6, 6, 9, 9]
[2, 7, 6, 0, 5, 4, 4, 2] -> [0, 2, 2, 4, 4, 5, 6, 7]


## Two weaknesses
**1. Length generalisation.** The model only ever saw length-8 inputs, so it fails at *any* other length, even shorter ones.

In [5]:
for Ltest in [4, 8, 12]:
    xt, yt = batch(500, Ltest)
    pred = model.generate(xt, Ltest + 1)
    print(f"test length {Ltest:2d}: token accuracy {(pred == yt).float().mean():.3f}")

test length  4: token accuracy 0.309
test length  8: token accuracy 1.000
test length 12: token accuracy 0.290


**2. The fixed-size bottleneck.** The encoder has to squeeze the whole input into one vector. Train fresh models (same size, same budget) on longer sequences: the longer the input, the harder it gets. Cho et al. saw translation quality drop sharply with sentence length for exactly this reason. **Attention** (next notebook) removes the bottleneck.

In [6]:
for Ltrain in [8, 16, 24]:
    seed_everything(0)
    m = Seq2Seq(V).to(device); o = torch.optim.Adam(m.parameters(), lr=2e-3)
    for step in range(1500):
        x, y = batch(128, Ltrain)
        loss = F.cross_entropy(m(x, y).reshape(-1, V), y.reshape(-1))
        o.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); o.step()
    xt, yt = batch(500, Ltrain); pred = m.generate(xt, Ltrain + 1)
    print(f"trained & tested on length {Ltrain:2d}: token acc {(pred == yt).float().mean():.3f} | full-sequence acc {(pred == yt).all(1).float().mean():.3f}")

trained & tested on length  8: token acc 0.996 | full-sequence acc 0.976


trained & tested on length 16: token acc 0.999 | full-sequence acc 0.990


trained & tested on length 24: token acc 0.884 | full-sequence acc 0.050
